<a href="https://colab.research.google.com/github/Wuanzz/FER-Video-Emotion-Recognition/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1: Cài đặt các thư viện cần thiết

In [225]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless -q
!pip install -q opencv-python-headless==4.10.0.84 gradio pandas ffmpeg-python

print("========================================")
print("Cài đặt thư viện hoàn tất!")
print("========================================")

Cài đặt thư viện hoàn tất!


# Cell 2: Kết nối Google Drive

In [226]:
from google.colab import drive

drive.mount('/content/drive')
print("Đã kết nối Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã kết nối Google Drive.


# Cell 3: Import thư viện & Kiểm tra Môi trường

In [227]:
import os
import cv2
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Device: {device}")

PyTorch Device: cuda


# Cell 4: Cấu hình Dự án & Đường dẫn File Checkpoint

In [228]:
PROJECT_DIR = "/content/drive/MyDrive/Công nghệ phần mềm nâng cao/Project_FER_Video"
MODEL_PATH = os.path.join(PROJECT_DIR, "checkpoints", "best_model.pth")

CLASS_NAMES = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
IMAGE_SIZE = (224, 224)
SEQUENCE_LENGTH = 16

# Cell 5: Khởi tạo OpenCV Haar Cascade Face Detector

In [229]:
cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(cascade_path)

if face_cascade.empty():
    raise RuntimeError("Không thể tải file XML Haar Cascade!")
print("Haar Cascade Face Detector sẵn sàng.")

Haar Cascade Face Detector sẵn sàng.


# Cell 6: Định nghĩa Kiến trúc Mô hình FER

In [230]:
class SpatialTemporalFERModel(nn.Module):
    def __init__(self, backbone_name="resnet18", rnn_type="LSTM", hidden_dim=256,
                 num_rnn_layers=2, dropout=0.5, num_classes=8):
        super().__init__()

        if backbone_name == "resnet18":
            resnet = models.resnet18(weights=None)
            feature_dim = resnet.fc.in_features
            resnet.fc = nn.Identity()
            self.backbone = resnet
        elif backbone_name == "mobilenet_v2":
            mobilenet = models.mobilenet_v2(weights=None)
            feature_dim = mobilenet.classifier[1].in_features
            mobilenet.classifier = nn.Identity()
            self.backbone = mobilenet

        self.rnn_type = rnn_type
        if rnn_type == "LSTM":
            self.rnn = nn.LSTM(input_size=feature_dim, hidden_size=hidden_dim,
                               num_layers=num_rnn_layers, batch_first=True,
                               bidirectional=True, dropout=dropout if num_rnn_layers > 1 else 0)
        elif rnn_type == "GRU":
            self.rnn = nn.GRU(input_size=feature_dim, hidden_size=hidden_dim,
                              num_layers=num_rnn_layers, batch_first=True,
                              bidirectional=True, dropout=dropout if num_rnn_layers > 1 else 0)

        rnn_out_dim = hidden_dim * 2
        self.classifier = nn.Sequential(
            nn.Linear(rnn_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        if x.dim() == 5 and x.shape[-1] == 3:
            x = x.permute(0, 1, 4, 2, 3)

        batch_size, seq_len, C, H, W = x.shape
        x = x.reshape(batch_size * seq_len, C, H, W)
        cnn_features = self.backbone(x)

        rnn_input = cnn_features.reshape(batch_size, seq_len, -1)
        rnn_output, _ = self.rnn(rnn_input)
        temporal_feature = torch.mean(rnn_output, dim=1)

        logits = self.classifier(temporal_feature)
        return logits

# Cell 7: Tải Checkpoint Mô hình

In [231]:
checkpoint = torch.load(MODEL_PATH, map_location="cpu")
MODEL_CONFIG = checkpoint["config"]

model = SpatialTemporalFERModel(
    backbone_name=MODEL_CONFIG["backbone"],
    rnn_type=MODEL_CONFIG["rnn_type"],
    hidden_dim=MODEL_CONFIG["hidden_dim"],
    num_rnn_layers=MODEL_CONFIG["num_rnn_layers"],
    dropout=MODEL_CONFIG["dropout"],
    num_classes=MODEL_CONFIG["num_classes"]
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()
print("Đã nạp thành công mô hình FER!")

Đã nạp thành công mô hình FER!


# Cell 8: Bộ xử lý Frame Buffer & Dự đoán Realtime/Batch

In [232]:
import time
import subprocess
import matplotlib.pyplot as plt

model_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Các biến phục vụ luồng Webcam Realtime Stream
frame_buffer = []
timeline_stream_history = []
last_process_time = 0
last_pred_emotion = "NEUTRAL"
last_pred_conf = 0.0
last_chart_fig = None

def process_single_frame_fast(input_frame):
    """
    Xử lý Realtime Streaming cho chế độ Webcam Live
    """
    global frame_buffer, timeline_stream_history, last_process_time
    global last_pred_emotion, last_pred_conf, last_chart_fig

    if input_frame is None:
        return None, "CHO WEBCAM...", 0.0, pd.DataFrame(), None

    current_time = time.time()

    if (current_time - last_process_time) < 0.25 and len(frame_buffer) > 0:
        gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.2, 4, minSize=(40, 40))
        annotated = input_frame.copy()
        for (x, y, w, h) in faces:
            cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(annotated, f"{last_pred_emotion} ({last_pred_conf:.1f}%)",
                        (x, max(20, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        df_hist = pd.DataFrame(timeline_stream_history[-10:]) if timeline_stream_history else pd.DataFrame()
        return annotated, last_pred_emotion, round(last_pred_conf, 2), df_hist, last_chart_fig

    last_process_time = current_time

    gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(40, 40))

    annotated_frame = input_frame.copy()
    cropped_face = input_frame.copy()

    if len(faces) > 0:
        x, y, w, h = max(faces, key=lambda r: r[2] * r[3])
        px, py = int(0.1 * w), int(0.1 * h)
        x1, y1 = max(0, x - px), max(0, y - py)
        x2, y2 = min(input_frame.shape[1], x + w + px), min(input_frame.shape[0], y + h + py)

        cropped_face = input_frame[y1:y2, x1:x2]
        cv2.rectangle(annotated_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    rgb_face = cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB)
    resized_face = cv2.resize(rgb_face, IMAGE_SIZE)

    frame_buffer.append(resized_face)
    if len(frame_buffer) > SEQUENCE_LENGTH:
        frame_buffer.pop(0)

    temp_sequence = list(frame_buffer)
    while len(temp_sequence) < SEQUENCE_LENGTH:
        temp_sequence.append(resized_face)

    tensors = [model_transform(f) for f in temp_sequence]
    video_tensor = torch.stack(tensors, dim=0).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(video_tensor)
        probs = torch.softmax(logits, dim=1)[0]

    top_idx = torch.argmax(probs).item()
    last_pred_emotion = CLASS_NAMES[top_idx].upper()
    last_pred_conf = float(probs[top_idx].item() * 100.0)

    if len(faces) > 0:
        cv2.putText(annotated_frame, f"{last_pred_emotion} ({last_pred_conf:.1f}%)",
                    (x, max(20, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    sec_str = time.strftime("%H:%M:%S", time.localtime())
    timeline_stream_history.append({
        "Mốc thời gian": sec_str,
        "Cảm xúc (Emotion)": last_pred_emotion,
        "Độ tin cậy": f"{last_pred_conf:.1f}%"
    })
    df_timeline = pd.DataFrame(timeline_stream_history[-10:])

    fig, ax = plt.subplots(figsize=(6, 2.8))
    all_probs = [probs[i].item() * 100 for i in range(len(CLASS_NAMES))]
    bars = ax.bar(CLASS_NAMES, all_probs, color='#3182bd')
    bars[top_idx].set_color('#e6550d')
    ax.set_ylim(0, 100)
    ax.set_ylabel("Độ tin cậy (%)")
    ax.tick_params(axis="x", rotation=35, labelsize=8)
    plt.tight_layout()
    last_chart_fig = fig

    return annotated_frame, last_pred_emotion, round(last_pred_conf, 2), df_timeline, last_chart_fig


def process_uploaded_video(video_path):
    """
    Hàm xử lý Video Upload / Record Webcam - Đã sửa lỗi HTML5 Video & Lỗi Mốc thời gian 0s
    """
    if video_path is None:
        return None, "CHUA NAP FILE", 0.0, pd.DataFrame(), None

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, "LOI MO FILE", 0.0, pd.DataFrame(), None

    # 1. Đọc toàn bộ frames
    all_frames = []
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            break
        all_frames.append(frame)
    cap.release()

    total_frames_count = len(all_frames)
    if total_frames_count == 0:
        return None, "VIDEO RONG", 0.0, pd.DataFrame(), None

    # 2. Xử lý FPS & Thời lượng chuẩn
    cap_temp = cv2.VideoCapture(video_path)
    fps_prop = cap_temp.get(cv2.CAP_PROP_FPS)
    cap_temp.release()

    if fps_prop <= 0 or np.isnan(fps_prop) or fps_prop > 120:
        fps_prop = 25.0

    duration_sec = total_frames_count / fps_prop
    height, width, _ = all_frames[0].shape

    # 3. Trích xuất 16 frames cho mô hình FER
    sample_indices = np.linspace(0, total_frames_count - 1, SEQUENCE_LENGTH, dtype=int)
    extracted_faces = []

    for idx in sample_indices:
        frame = all_frames[idx]
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))

        if len(faces) > 0:
            x, y, w, h = max(faces, key=lambda r: r[2] * r[3])
            px, py = int(0.1 * w), int(0.1 * h)
            x1, y1 = max(0, x - px), max(0, y - py)
            x2, y2 = min(width, x + w + px), min(height, y + h + py)
            crop = frame[y1:y2, x1:x2]
        else:
            crop = frame

        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        extracted_faces.append(cv2.resize(crop, IMAGE_SIZE))

    tensors = [model_transform(f) for f in extracted_faces]
    video_tensor = torch.stack(tensors, dim=0).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(video_tensor)
        probs = torch.softmax(logits, dim=1)[0]

    top_idx = torch.argmax(probs).item()
    pred_emotion = CLASS_NAMES[top_idx].upper()
    pred_conf = float(probs[top_idx].item() * 100.0)

    # 4. Ghi file video tạm bằng cv2
    raw_output_path = "/content/temp_raw_output.mp4"
    web_output_path = "/content/annotated_web_output.mp4"

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(raw_output_path, fourcc, fps_prop, (width, height))

    records = []
    step_log = max(int(total_frames_count / 10), 1)

    for fc, frame in enumerate(all_frames):
        annotated_frame = frame.copy()
        current_sec = fc / fps_prop  # Sửa công thức mốc thời gian chuẩn xác

        gray = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))

        for (x, y, w, h) in faces:
            cv2.rectangle(annotated_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(annotated_frame, f"{pred_emotion} ({pred_conf:.1f}%)",
                        (x, max(20, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        out.write(annotated_frame)

        if fc % step_log == 0 or fc == total_frames_count - 1:
            records.append({
                "Mốc thời gian": f"{current_sec:.1f}s / {duration_sec:.1f}s",
                "Cảm xúc (Emotion)": pred_emotion,
                "Độ tin cậy": f"{pred_conf:.1f}%"
            })

    out.release()

    # 5. CONVERT SANG CODEC H.264 ĐỂ BẢO ĐẢM XEM ĐƯỢC TRÊN GRADIO/TRÌNH DUYỆT WEB
    try:
        subprocess.run([
            "ffmpeg", "-y", "-i", raw_output_path,
            "-vcodec", "libx264", "-crf", "23", web_output_path
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        final_video = web_output_path
    except Exception:
        final_video = raw_output_path  # Fallback nếu không convert được ffmpeg

    df_timeline = pd.DataFrame(records)

    # 6. Biểu đồ xác suất
    fig, ax = plt.subplots(figsize=(6, 2.8))
    all_probs = [probs[i].item() * 100 for i in range(len(CLASS_NAMES))]
    bars = ax.bar(CLASS_NAMES, all_probs, color='#3182bd')
    bars[top_idx].set_color('#e6550d')
    ax.set_ylim(0, 100)
    ax.set_ylabel("Độ tin cậy (%)")
    ax.tick_params(axis="x", rotation=35, labelsize=8)
    plt.tight_layout()

    return final_video, pred_emotion, round(pred_conf, 2), df_timeline, fig

print("Đã cập nhật xong bộ xử lý video!")

Đã cập nhật xong bộ xử lý video!


# Cell 9: Giao diện Web Bố cục 3 Phần UX Chuẩn

In [233]:
import gradio as gr

with gr.Blocks(title="Facial Emotion Recognition Realtime Dashboard") as demo:

    gr.Markdown(
        """
        # Hệ thống Nhận diện Biểu cảm Khuôn mặt (FER)
        ### Đồ án Công nghệ Phần mềm Nâng cao — Multi-Mode Emotion Analytics
        """
    )

    # PHẦN 1: BẢNG NẠP ĐẦU VÀO
    with gr.Accordion(label="", open=True):
        gr.Markdown("### PHẦN 1: BẬT MÁY ẢNH WEBCAM HOẶC CHỌN FILE VIDEO")

        mode_radio = gr.Radio(
            choices=[
                "Quay Webcam Ngắn (Record 3s - Khuyên dùng)",
                "Tải Tập Tin Video (.mp4)",
                "Webcam Live Trực Tiếp (Realtime Stream)"
            ],
            value="Quay Webcam Ngắn (Record 3s - Khuyên dùng)",
            label="Lựa chọn chế độ nhận diện:",
            interactive=True
        )

        with gr.Row():
            with gr.Column(scale=1):
                process_btn = gr.Button("Phân tích Video / Clip Quay", variant="primary")

            with gr.Column(scale=2):
                webcam_record_input = gr.Video(
                    sources=["webcam"],
                    label="Quay đoạn clip ngắn 3-5 giây",
                    visible=True,
                    height=220
                )
                file_input = gr.Video(
                    sources=["upload"],
                    label="Tải tập tin video (.mp4)",
                    visible=False,
                    height=220
                )
                webcam_stream_input = gr.Image(
                    sources=["webcam"],
                    streaming=True,
                    label="Luồng Live Stream Webcam",
                    visible=False,
                    height=220
                )

    # PHẦN DƯỚI: MÀN HÌNH THEO DỎI VÀ ANALYTICS
    with gr.Row():

        # PHẦN 2: MÀN HÌNH KẾT QUẢ (LEFT)
        with gr.Column(scale=1):
            gr.Markdown("### PHẦN 2: Màn Hình Luồng Video / Output")
            live_image_output = gr.Image(label="Live Stream Overlay Bounding Box", visible=False, height=320)
            file_video_output = gr.Video(label="Video kết quả có Bounding Box", visible=True, height=320)

        # PHẦN 3: BẢNG THÔNG TIN BÁO CÁO (RIGHT)
        with gr.Column(scale=1):
            gr.Markdown("### PHẦN 3: Bảng Theo Dõi & Analytics")

            with gr.Row():
                emotion_output = gr.Textbox(label="Cảm xúc nhận diện")
                confidence_output = gr.Number(label="Độ tin cậy (%)", precision=2)

            timeline_table = gr.Dataframe(
                headers=["Mốc thời gian", "Cảm xúc (Emotion)", "Độ tin cậy"],
                label="Nhật ký phân tích",
                interactive=False
            )

            chart_output = gr.Plot(label="Xác suất 8 lớp cảm xúc")

    # CHUYỂN ĐỔI GIAO DIỆN CHUẨN
    def switch_mode(choice):
        if choice == "Quay Webcam Ngắn (Record 3s - Khuyên dùng)":
            return (
                gr.update(visible=True), gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=True), gr.update(visible=False), gr.update(visible=True)
            )
        elif choice == "Tải Tập Tin Video (.mp4)":
            return (
                gr.update(visible=False), gr.update(visible=True), gr.update(visible=False),
                gr.update(visible=True), gr.update(visible=False), gr.update(visible=True)
            )
        else:
            return (
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=True),
                gr.update(visible=False), gr.update(visible=True), gr.update(visible=False)
            )

    mode_radio.change(
        fn=switch_mode,
        inputs=mode_radio,
        outputs=[
            webcam_record_input, file_input, webcam_stream_input,
            process_btn, live_image_output, file_video_output
        ]
    )

    # SỰ KIỆN NÚT BẤM (DÀNH CHO MODE RECORD & UPLOAD)
    def handle_process(mode, webcam_vid, file_vid):
        if mode == "Quay Webcam Ngắn (Record 3s - Khuyên dùng)":
            return process_uploaded_video(webcam_vid)
        else:
            return process_uploaded_video(file_vid)

    process_btn.click(
        fn=handle_process,
        inputs=[mode_radio, webcam_record_input, file_input],
        outputs=[file_video_output, emotion_output, confidence_output, timeline_table, chart_output]
    )

    # SỰ KIỆN STREAMING (DÀNH CHO MODE REALTIME STREAM)
    webcam_stream_input.stream(
        fn=process_single_frame_fast,
        inputs=webcam_stream_input,
        outputs=[live_image_output, emotion_output, confidence_output, timeline_table, chart_output],
        stream_every=0.25
    )

print("Đã khởi tạo giao diện chuẩn hóa!")

Đã khởi tạo giao diện chuẩn hóa!


# Cell 10: Khởi chạy Web Demo

In [234]:
print("Đang khởi chạy ứng dụng Web Demo...")
demo.launch(
    theme=gr.themes.Soft(),
    share=True,
    debug=True
)

Đang khởi chạy ứng dụng Web Demo...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1bbbb58a040a8dbb69.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1bbbb58a040a8dbb69.gradio.live
